> ## SOLUTION / ANSWER KEY &mdash; Lab 7.1
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-01-the-automation-pipeline.ipynb`](../lab-01-the-automation-pipeline.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 7.1 &mdash; The Automation Pipeline

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Build the six pipeline stages as a real StateGraph, wired in order
- See where the Day-3 loop lives (inside the draft node)
- Mark the approval checkpoint that guards the irreversible act

> **How this lab works (near-real):** you have a `GROQ_API_KEY` in the repo `.env`. Read the **Concept**, fill the real `___` blanks in **Build it** (real pipeline logic, real tool bodies, the real draft/`create_agent` call), then **Run it for real** &mdash; a real Groq model drives the step over real tools &mdash; and **read the output/trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working email agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`) and a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")` &mdash; reliable tool-calling via `create_agent`). If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **email-drafting agent** (the client's Lab 4.1): it **drafts but never sends** &mdash; `send_email` is withheld and a human approves.

**Reference:** [Module 7 slides &mdash; The task-automation pipeline](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, time, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (+ other keys)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-07-01")
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is configured -- the 'Run it for real' cells self-skip if not."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
MODEL = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL, temperature=0) if groq_ready() else None

def with_backoff(fn, tries=4):
    """Run fn(); retry with backoff on Groq rate limits (HTTP 429). Other errors propagate."""
    last = None
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if "429" in str(e) or "rate limit" in str(e).lower() or "rate_limit" in str(e).lower():
                wait = 5 * (attempt + 1)
                print(f"(rate-limited -- retrying in {wait}s)")
                time.sleep(wait)
                continue
            raise
    raise last

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("Groq ready | model:", MODEL, "| WORK:", WORK)
else:
    print("GROQ_API_KEY not set -- add it to the repo .env (free key at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
Every task-automation agent, however complex, follows the **same pipeline** (deck slide 5):
**trigger** &rarr; **gather** &rarr; **draft** &rarr; **validate** &rarr; **approve** &rarr; **act**.
You build it here as a **real LangGraph `StateGraph`** &mdash; six nodes wired in order &mdash; the same
framework the later labs' real tools and model slot into. The Day-3 ReAct loop lives inside the *draft*
node; the outer stages (gather, validate, approve) are what make it **reliable** and **safe**. **approve**
is a human checkpoint guarding the one irreversible step: **act**. (Module 8 grows this straight line into
branches, parallel fan-out and reducers.)

In [ ]:
STAGES = ["trigger", "gather", "draft", "validate", "approve", "act"]
print("the shape of every automation:")
print(" -> ".join(STAGES))

## Build it
Build the pipeline as a real `StateGraph`: wire each stage node to the next in order, and mark the human
checkpoint. `make_stage` and the state schema are written for you.

In [ ]:
from typing import Annotated, TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END

STAGES = ["trigger", "gather", "draft", "validate", "approve", "act"]

class PipelineState(TypedDict):
    trail: Annotated[list, add]        # each stage node appends its name (a reducer)

def make_stage(name):
    """Each stage is a node that records it ran (later labs put real tools/model inside)."""
    def node(state):
        return {"trail": [name]}
    return node

def build_pipeline():
    g = StateGraph(PipelineState)
    for s in STAGES:
        g.add_node(s, make_stage(s))
    g.add_edge(START, "trigger")
    for a, b in zip(STAGES, STAGES[1:]):
        g.add_edge(a, b)
    g.add_edge(STAGES[-1], END)
    return g.compile()

def is_checkpoint(stage):
    # the human approval gate: a person must approve before the irreversible act
    return stage == "approve"

try:
    app = build_pipeline()
    final = app.invoke({"trail": []})
    print("pipeline ran:", " -> ".join(final["trail"]))
    print("graph nodes :", sorted(set(app.get_graph().nodes) - {"__start__", "__end__"}))
    print("checkpoint at approve?", is_checkpoint("approve"))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- `build_pipeline()` compiles a **real `StateGraph`**; running it walks all six nodes in order &mdash; that ordering is the contract every later lab honours.
- `is_checkpoint("approve")` marks the one **human** gate; everything before it is autonomous, everything after is irreversible.
- The `draft` node is where the real model runs (Lab 6); `gather` is where real tools run (Lab 2). Module 8 turns this line into branches and parallel fan-out.

## Your turn (open task &mdash; no grader)
Add a seventh stage node &mdash; e.g. `"log"` after `act` &mdash; wire it in, and re-run. **What good looks
like:** the walk includes your new node in the right place and the graph still terminates at `END`. Then ask:
which stages are reversible, and which one truly needs the human gate?

In [ ]:
# --- Reference answer (ONE good way to do the 'Your turn' task -- compare with your own) ---
STAGES.append("log")                             # a 7th stage node, after "act"
final = build_pipeline().invoke({"trail": []})   # STAGES[-1] is now "log" -> wired to END automatically
print("pipeline ran:", " -> ".join(final["trail"]))   # includes "log", still terminates
print("checkpoint at approve?", is_checkpoint("approve"))
print("reversible: log yes / act no -- only the pre-act 'approve' stage truly needs the human gate")

---
### Key takeaway
Trigger -> gather -> draft -> validate -> approve -> act, built as a real StateGraph. The outer stages are what turn a demo agent into an automation. Next: the gather stage -- grounding the task in real data with real tools.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>